# 어텐션 비묶음(untied) 루프 MoE — §3.3 재측정

**무엇을 왜**: 논문 §3.3의 sticky vs re-route 동률(Δ=+0.0032±0.0035, n=3)은
`model.py`의 `Block`이 **어텐션과 FFN을 한 모듈에 묶어** 통째로 R번 돌린 조건에서 나온 값이다.
§4가 "어텐션은 층별로 유지하고 전문가만 접는다"로 가면 그 수치는 적용 범위를 벗어난다.
이 노트북은 **어텐션을 반복마다 독립**으로 두고 같은 비교를 다시 돌려 그 구멍을 막는다.
(§4 fold 구성에 충실한 형태. [8]의 per-layer independent router 축은 미측정.)

**주 결과**: untied 조건의 짝지은 Δ(sticky − re-route), 시드 3개.
두 팔의 상주량이 정확히 같으므로 짝지은 비교가 성립한다(G-D가 검사).
tied 앵커 2런도 **같은 세션·같은 dtype으로 다시 측정**해서, 비교를 Δ 대 Δ로만 한다.

**사용법**: 런타임 유형 **L4 GPU** → 런타임 → 모두 실행. 드라이브 허용만 처리.
중단되면 그대로 다시 실행 — 완주한 런은 몇 초 만에 통과한다.

**출력 위치**: 이 세션은 `MyDrive/rdepth_untied/`에만 쓴다.
발표된 §3 로그(`rdepth_out/logs/`)는 건드리지 않는다 — `train.py:114`가 append 모드라
같은 디렉터리를 쓰면 발표 로그에 행이 붙는다.

**예산**: 8런 × 약 45분 ≈ 6시간 ≈ **29 CU** (L4 4.82 CU/h 가정).
[5]의 `run()`이 매 런 시작 전에 70 CU 상한을 검사하고, [11]이 런타임을 반납한다.
반납 셀을 빼먹으면 탭이 붙어 있는 시간만큼 계속 과금된다.

**선행 조건**: `MyDrive/rdepth_out/train.bin`이 있어야 한다(기존 §3 노트북이 만들어 둔 캐시).
없으면 [4]가 중단시킨다 — 20~30분짜리 CPU 데이터 준비를 GPU 요금으로 돌리지 않기 위해서다.

In [ ]:
# [0] 환경 점검
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, shutil
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 — 런타임 유형을 L4 GPU로 바꾸세요'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU={name}  VRAM={vram:.1f}GiB  cap={torch.cuda.get_device_capability()}  '
      f'disk_free={shutil.disk_usage("/content").free/2**30:.0f}GiB')
assert 'L4' in name or 'A100' in name, \
    f'{name}: T4에서는 8런이 세션 한도를 넘깁니다(약 4배 느림). 런타임 유형을 바꾸세요.'
print('\n컴퓨트 유닛 잔량은 우측 상단 리소스 패널에서 확인하세요 (API로 읽을 수 없습니다).')

In [ ]:
# [1] 드라이브 + 코드 + 세션 상수 (멱등)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, shutil, time, torch

DRIVE = '/content/drive/MyDrive/rdepth_out'      # 데이터 캐시 전용 — 이 세션은 여기에 쓰지 않는다
OUT   = '/content/drive/MyDrive/rdepth_untied'   # 이 세션의 모든 로그/체크포인트
os.makedirs(f'{OUT}/logs', exist_ok=True)
os.makedirs(f'{OUT}/ckpt', exist_ok=True)
os.environ['RDEPTH_OUT'] = OUT                   # 여기서 설정 — 뒤 셀을 건너뛰어도 안전
T_START = time.time()
DTYPE = 'bf16' if torch.cuda.get_device_capability()[0] >= 8 else 'fp16'
assert shutil.disk_usage(DRIVE).free / 2**30 > 8, 'Drive 여유 공간 부족 — ckpt 8개에 약 4.3GB 필요'
print(f'OUT={OUT}  DTYPE={DTYPE}')

if not os.path.exists('/content/rdepth/model.py'):
    r = subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/cornch-k/fold-dont-stream.git', '/content/fds'],
                       capture_output=True, text=True)
    if r.returncode == 0 and os.path.exists('/content/fds/training/rdepth/model.py'):
        subprocess.run(['cp', '-rT', '/content/fds/training/rdepth', '/content/rdepth'], check=True)
        sha = subprocess.run(['git', '-C', '/content/fds', 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip()
        print('GitHub에서 코드 확보. commit =', sha)
    else:
        import glob, zipfile
        z = sorted(glob.glob(f'{DRIVE}/rdepth_code*.zip'))
        assert z, ('클론 실패이고 드라이브에도 zip이 없습니다. '
                   f'{DRIVE}/rdepth_code_v4.zip 형태로 올려주세요.\n' + r.stderr[-500:])
        zipfile.ZipFile(z[-1]).extractall('/content/')
        assert os.path.exists('/content/rdepth/model.py'), 'zip 구조가 예상과 다릅니다'
        print('드라이브 zip에서 코드 확보:', z[-1])
!pip -q install tokenizers 2>/dev/null | tail -1

def run(name, seed=1337):
    """train.py를 돌린다. 실패하면 즉시 예외 — !python은 종료 코드를 삼켜서 6시간 뒤에야 발견된다.
       [5]~[8]이 각각 독립 재실행 가능하도록 여기서 정의한다([1]만 다시 돌리면 복구)."""
    spent = (time.time() - T_START) / 3600 * 4.82
    assert spent < 70, f'예산 상한: 이미 약 {spent:.0f} CU 소모. 남은 런은 다음 세션으로 미루세요.'
    t0 = time.time()
    subprocess.run(['python', 'train.py', '--run', name, '--max-tokens', '100000000',
                    '--dtype', DTYPE, '--micro-batch', '16', '--resume', '--seed', str(seed)],
                   cwd='/content/rdepth', check=True)
    print(f'[{name} seed={seed}] {(time.time()-t0)/60:.1f}분  (시작 시점 누적 약 {spent:.0f} CU)', flush=True)

print('준비 완료')

In [ ]:
# [2] model.py 패치: tie_attn (루프 반복 r마다 독립 어텐션 / MoE FFN·라우터·ln2는 공유)
#     측정 범위: untied attention / tied router / tied ln2.
#     이건 §4의 fold(FFN·전문가만 접기)에 충실한 구성이지,
#     참조 [8]의 per-layer independent router 구성이 아니다 — 라우터 축은 여기서 미측정.
import re, pathlib, hashlib

p = pathlib.Path('/content/rdepth/model.py')
BACKUP = pathlib.Path('/content/rdepth/model_orig.py')
if not BACKUP.exists():
    BACKUP.write_text(p.read_text(encoding='utf-8'), encoding='utf-8')
src = BACKUP.read_text(encoding='utf-8')   # 멱등: 항상 원본에서 다시 패치

def sub1(pat, rep, s, tag):
    out, n = re.subn(pat, rep, s, count=1)
    assert n == 1, f'패치 실패 [{tag}]: 앵커 {n}개 일치 (1이어야 함) — model.py 버전 확인 필요'
    return out

# (1) GPTConfig에 tie_attn 추가
src = sub1(
    r'(    route_policy: str = "rr"[^\n]*\n)',
    r'\1    tie_attn: bool = True   # False = 루프 반복 r마다 독립 어텐션 (FFN/MoE/ln2는 공유)\n',
    src, 'cfg')

# (2) AttnUnit 클래스 (r>=1 반복용 어텐션 뱅크)
src = sub1(
    r'(class Block\(nn\.Module\):\n)',
    'class AttnUnit(nn.Module):\n'
    '    """루프 반복 r>=1 전용 어텐션 파라미터. r=0은 Block 자신의 ln1/qkv/proj를 쓴다."""\n'
    '    def __init__(self, cfg):\n'
    '        super().__init__()\n'
    '        self.ln1 = RMSNorm(cfg.d)\n'
    '        self.qkv = nn.Linear(cfg.d, 3 * cfg.d, bias=False)\n'
    '        self.proj = nn.Linear(cfg.d, cfg.d, bias=False)\n'
    '\n'
    r'\1',
    src, 'attnunit')

# (3) Block.__init__ 시그니처 + 뱅크 생성 (n_attn=1이면 빈 ModuleList = 파라미터 0개)
src = sub1(
    r'    def __init__\(self, cfg\):\n        super\(\)\.__init__\(\)\n        self\.n_head, self\.d = cfg\.n_head, cfg\.d\n',
    '    def __init__(self, cfg, n_attn=1):\n'
    '        super().__init__()\n'
    '        self.n_head, self.d = cfg.n_head, cfg.d\n',
    src, 'block_init')
src = sub1(
    r'(        self\.proj = nn\.Linear\(cfg\.d, cfg\.d, bias=False\)\n)(        if cfg\.use_moe:)',
    r'\1        self.attn_x = nn.ModuleList([AttnUnit(cfg) for _ in range(max(0, n_attn - 1))])\n\2',
    src, 'block_bank')

# (4) Block.forward: r로 어텐션 뱅크 선택 (r=0 또는 tied면 self)
src = sub1(
    r'    def forward\(self, x, cache=None, reuse=False\):\n'
    r'        B, T, C = x\.shape\n'
    r'        h = self\.ln1\(x\)\n'
    r'        q, k, v = self\.qkv\(h\)\.split\(C, dim=2\)\n',
    '    def forward(self, x, cache=None, reuse=False, r=0):\n'
    '        B, T, C = x.shape\n'
    '        u = self.attn_x[r - 1] if (r > 0 and len(self.attn_x) >= r) else self\n'
    '        h = u.ln1(x)\n'
    '        q, k, v = u.qkv(h).split(C, dim=2)\n',
    src, 'block_fwd')
src = sub1(
    r'        x = x \+ self\.proj\(a\.transpose\(1, 2\)\.contiguous\(\)\.view\(B, T, C\)\)\n',
    '        x = x + u.proj(a.transpose(1, 2).contiguous().view(B, T, C))\n',
    src, 'block_proj')

# (5) loop_block만 뱅크를 갖는다 (pre/post는 1회 실행이므로 n_attn=1)
src = sub1(
    r'        self\.loop_block = nn\.ModuleList\(\[Block\(cfg\) for _ in range\(cfg\.n_loop_layers\)\]\)\n',
    '        self.loop_block = nn.ModuleList([Block(cfg, n_attn=(1 if getattr(cfg, "tie_attn", True) else cfg.n_loops))\n'
    '                                         for _ in range(cfg.n_loop_layers)])\n',
    src, 'gpt_loopblock')

# (6) forward 루프에서 r 전달
src = sub1(
    r'                x = b\(x, cache=cache, reuse=\(r > 0\)\)\n',
    '                x = b(x, cache=cache, reuse=(r > 0), r=r)\n',
    src, 'gpt_fwd')

# (7) 새 CONFIGS 두 개
src = sub1(
    r'(                             n_pre=1, n_loop_layers=2, n_loops=3, n_post=1, route_policy="sticky"\),\n)\}',
    r'\1'
    '    # [untied-attn] 루프 반복마다 독립 어텐션 + 공유 MoE FFN/라우터/ln2\n'
    '    "moe-loop-st-ut": GPTConfig(d=384, n_head=6, use_moe=True, route_policy="sticky",\n'
    '                                n_pre=1, n_loop_layers=2, n_loops=3, n_post=1, tie_attn=False),\n'
    '    "moe-loop-rr-ut": GPTConfig(d=384, n_head=6, use_moe=True, route_policy="rr",\n'
    '                                n_pre=1, n_loop_layers=2, n_loops=3, n_post=1, tie_attn=False),\n'
    '}',
    src, 'configs')

p.write_text(src, encoding='utf-8')
print('model.py 패치 완료. sha256 =', hashlib.sha256(src.encode()).hexdigest()[:16])

# (8) train.py: 체크포인트 원자적 저장. 531MB를 Drive FUSE에 in-place로 쓰다가 선점되면
#     이후 모든 --resume이 예외 = 유닛만 쓰고 산출물 0. os.replace가 FUSE에서 POSIX 원자성을
#     보장하진 않지만 in-place 덮어쓰기보다는 엄격히 낫다.
tp = pathlib.Path('/content/rdepth/train.py')
TBACKUP = pathlib.Path('/content/rdepth/train_orig.py.txt')
if not TBACKUP.exists():
    TBACKUP.write_text(tp.read_text(encoding='utf-8'), encoding='utf-8')
ts = TBACKUP.read_text(encoding='utf-8')   # 멱등: 항상 원본에서 다시 패치
ts2 = ts.replace('"step": step, "gen": gen.get_state()}, ckpt_path)',
                 '"step": step, "gen": gen.get_state()}, ckpt_path + ".tmp")\n'
                 '            os.replace(ckpt_path + ".tmp", ckpt_path)')
assert ts2 != ts, 'train.py 저장 앵커 불일치 — 버전 확인 필요'
tp.write_text(ts2, encoding='utf-8')
print('train.py 원자적 저장 패치 완료')

In [ ]:
# [3] 무해성 게이트 — 이게 통과하지 못하면 아래 결과는 전부 무효다.
#   G-A: tie_attn=True 경로가 패치 전과 "비트 동일"인가 (같은 시드 → 같은 파라미터, 같은 로짓)
#   G-B: tie_attn=False가 어텐션 파라미터만 정확히 (R-1)×n_loop_layers만큼 늘리는가
#   G-C: 늘어난 파라미터가 실제로 forward에 쓰이는가 (grad가 흐르는가)
#   G-D: st/rr 두 untied 구성의 상주량이 같은가 (짝지은 비교의 전제)
import sys, torch
sys.path.insert(0, '/content/rdepth')

def fresh(mod_path):
    import importlib.util, uuid
    spec = importlib.util.spec_from_file_location('m_' + uuid.uuid4().hex[:8], mod_path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

orig, patched = fresh('/content/rdepth/model_orig.py'), fresh('/content/rdepth/model.py')

torch.manual_seed(1337); a = orig.GPT(orig.CONFIGS['moe-loop-st'])
torch.manual_seed(1337); b = patched.GPT(patched.CONFIGS['moe-loop-st'])
sa, sb = a.state_dict(), b.state_dict()
assert set(sa) == set(sb), f'G-A 실패: 텐서 집합 불일치 {set(sa) ^ set(sb)}'
for k in sa:
    assert torch.equal(sa[k], sb[k]), f'G-A 실패: {k} 초기값 불일치 (RNG 순서가 바뀜)'
x = torch.randint(0, 4096, (2, 128))
a.eval(); b.eval()
with torch.no_grad():
    la, _ = a(x); lb, _ = b(x)
assert torch.equal(la, lb), 'G-A 실패: tied 경로 로짓 불일치'
P_TIED = a.num_unique_params()
print(f'G-A PASS  tied 경로 비트 동일 (파라미터 {P_TIED:,})')

torch.manual_seed(1337); c = patched.GPT(patched.CONFIGS['moe-loop-st-ut'])
d_, R, nl = 384, 3, 2
per_attn = d_ + 3 * d_ * d_ + d_ * d_          # ln1 + qkv + proj
expect = P_TIED + nl * (R - 1) * per_attn
P_UNTIED = c.num_unique_params()
assert P_UNTIED == expect, f'G-B 실패: 기대 {expect:,} != 실측 {P_UNTIED:,} (차이 {P_UNTIED-expect:+,})'
print(f'G-B PASS  untied {P_UNTIED:,} (+{P_UNTIED-P_TIED:,}, +{100*(P_UNTIED/P_TIED-1):.2f}% / '
      f'상주 {c.resident_bytes()/1e6:.1f}MB)')

c.train()
_, loss = c(x, x)
loss.backward()
banks = [(n, q) for n, q in c.named_parameters() if '.attn_x.' in n]
assert banks, 'G-C 실패: attn_x 파라미터가 없음'
dead = [n for n, q in banks if q.grad is None or q.grad.abs().sum().item() == 0]
assert not dead, f'G-C 실패: grad가 0인 뱅크 {dead[:3]}'
print(f'G-C PASS  attn_x 뱅크 {len(banks)}개 전부 grad 수신')

torch.manual_seed(1337); e = patched.GPT(patched.CONFIGS['moe-loop-rr-ut'])
assert e.num_unique_params() == P_UNTIED, 'G-D 실패: st/rr 상주량 불일치'
print('G-D PASS  st/rr 상주량 동일 — 짝지은 비교 성립')
del a, b, c, e

In [ ]:
# [4] 데이터 — 드라이브 캐시에서만 가져온다.
#     GPU 런타임에서 20~30분짜리 CPU 작업(TinyStories 다운로드 + 4k BPE)을 돌리지 않는다.
import os
os.chdir('/content/rdepth')
os.makedirs('data', exist_ok=True)
DRIVE = '/content/drive/MyDrive/rdepth_out'
if os.path.exists(f'{DRIVE}/train.bin'):
    !cp {DRIVE}/tok4096.json {DRIVE}/val.bin {DRIVE}/train.bin data/
    print('드라이브 캐시 사용')
else:
    raise AssertionError(
        f'{DRIVE}/train.bin 이 없습니다. CPU 런타임(GPU 미할당)에서 prepare_data.py를 먼저 돌리고 '
        'data/{tok4096.json,val.bin,train.bin}을 이 경로에 복사한 뒤 다시 시작하세요.')
import numpy as np
tr, va = np.memmap('data/train.bin', dtype=np.uint16, mode='r'), np.memmap('data/val.bin', dtype=np.uint16, mode='r')
print(f'train={len(tr):,} val={len(va):,} 토큰')
assert len(tr) > 4e8, f'train.bin이 짧습니다({len(tr):,}) — 논문은 480.7M 토큰입니다'

In [ ]:
# [5] tied sticky 앵커 — 이 세션의 Δ_tied 절반.
#     발표 로그와 다른 디렉터리(OUT)이므로 이건 no-op이 아니라 실제 학습이다.
run('moe-loop-st')

In [ ]:
# [6] untied sticky × 3시드
for s in (1337, 1338, 1339):
    run('moe-loop-st-ut', s)

In [ ]:
# [7] untied re-route × 3시드
for s in (1337, 1338, 1339):
    run('moe-loop-rr-ut', s)

In [ ]:
# [8] tied re-route 앵커 — 이 세션 Δ_tied의 나머지 반쪽.
#     건너뛰면 주 결과의 비교자가 사라진다. 선택 사항이 아니다.
run('moe-loop-rr')

In [ ]:
# [9] 예산 대조 — 이 세션 디렉터리의 로그만 합산
import csv, os, glob
OUT = '/content/drive/MyDrive/rdepth_untied'
CU_PER_HOUR = 4.82   # L4 추정치. 리소스 패널의 실제 소모와 대조할 것.
hours = 0.0
for p in sorted(glob.glob(f'{OUT}/logs/*.csv')):
    rows = list(csv.DictReader(open(p)))
    if rows:
        h = float(rows[-1]['elapsed_s']) / 3600
        hours += h
        print(f'  {os.path.basename(p)[:-4]:<20} step={rows[-1]["step"]:>5} {h:.2f}h')
print(f'\n로그 기준 누적 {hours:.2f}h → 약 {hours*CU_PER_HOUR:.1f} CU')
print(f'벽시계 기준(재개분 포함) {(__import__("time").time()-T_START)/3600:.2f}h → '
      f'약 {(__import__("time").time()-T_START)/3600*CU_PER_HOUR:.1f} CU  ← 이쪽이 실제 과금에 가깝다')
if hours * CU_PER_HOUR > 60:
    print('!! 60 CU 초과. --max-tokens는 절대 낮추지 마세요(비교 무효). 남은 시드를 다음 세션으로 미루세요.')

In [ ]:
# [10] 판정 — untied 조건에서 sticky vs re-route
import csv, os, statistics as st
OUT = '/content/drive/MyDrive/rdepth_untied'
MAX_STEPS = 100_000_000 // 32768          # 3051 — 완주 기준
MARGIN = 0.010                            # nats. §4 설계 판단을 바꿀 최소 효과 (회복 분모 0.091의 약 11%p)

def _rows(run, seed=1337):
    p = f'{OUT}/logs/{run}{"" if seed == 1337 else f"-s{seed}"}.csv'
    return list(csv.DictReader(open(p))) if os.path.exists(p) else []

def best(run, seed=1337):
    """완주한 런만 값을 돌려준다. 중단된 팔과 완주한 팔을 짝지으면 Δ 전체가 삼켜진다."""
    rows = _rows(run, seed)
    if not rows or int(rows[-1]['step']) < MAX_STEPS:
        return None
    assert not any(r['val_loss'] in ('', 'nan') for r in rows), f'{run}(seed={seed}): val_loss에 nan — 발산한 런입니다'
    return min(float(r['val_loss']) for r in rows)

def touches(run, seed=1337):
    rows = _rows(run, seed)
    return rows[-1]['uniq_exp'] if rows else ''

SEEDS = (1337, 1338, 1339)
stv = [best('moe-loop-st-ut', s) for s in SEEDS]
rrv = [best('moe-loop-rr-ut', s) for s in SEEDS]
pairs = [(u, v, s) for u, v, s in zip(stv, rrv, SEEDS) if u is not None and v is not None]

print(f'{"seed":>6} {"st-ut":>9} {"rr-ut":>9} {"st-rr":>9}   (완주한 시드쌍만)')
for u, v, s in pairs:
    print(f'{s:>6} {u:>9.4f} {v:>9.4f} {u-v:>+9.4f}')
print(f'\n전문가 접촉 수: st-ut={touches("moe-loop-st-ut")} rr-ut={touches("moe-loop-rr-ut")}  '
      f'(논문 tied: 2.000 / 2.369)')

if len(pairs) < 3:
    print(f'\n!! 완주한 시드쌍 {len(pairs)}개 — 헤드라인 Δ는 계산하지 않습니다. [6],[7]을 다시 실행하세요.')
d = [u - v for u, v, _ in pairs] if len(pairs) >= 3 else []
if d:
    m = sum(d) / len(d)
    sd = st.stdev(d)
    print(f'\nuntied Δ(sticky−reroute) = {m:+.4f} ± {sd:.4f} nats  (n={len(d)}, 짝지은 차이의 표본표준편차)')
    print('\n판정 규칙(사전 등록):')
    print(f'  · |Δ| + sd < {MARGIN} → 동률.   |Δ| − sd > {MARGIN} → §3.3 재서술 필요.   그 사이 → 판정 불가(n 부족).')
    print('  · 구간 겹침은 동등성의 증거가 아닙니다. 효과크기와 사전 등록 마진으로만 판정합니다.')

anchor, tied_rr = best('moe-loop-st'), best('moe-loop-rr')
if anchor is not None and tied_rr is not None:
    dt = anchor - tied_rr
    print(f'\n이 세션 tied: st={anchor:.4f} rr={tied_rr:.4f} → Δ_tied = {dt:+.4f}')
    print(f'재현성 바닥: Δ_tied(이 세션, 1337) 대 Table 2 Δ(+0.0048) 차이 = {dt-0.0048:+.4f}')
    print('  이 차이가 |Δ_untied|보다 크면 결론은 "동률"이 아니라 "해상도 미만"입니다.')
elif anchor is not None:
    print(f'\n앵커(tied st) = {anchor:.4f} — tied rr이 없어 Δ_tied를 만들 수 없습니다. [8]을 실행하세요.')

print('\n' + '=' * 72)
print('보고 시 반드시 함께 적을 것 (핸드오프 CSV만 보는 3주 뒤의 자신을 위해):')
print(f'  1. untied 팔은 tied 대비 +2,360,832 params (+5.63%). tied↔untied 절대 손실 직접 비교 금지 —')
print(f'     비교 가능한 것은 Δ 대 Δ뿐입니다.')
print( '  2. 이 실험이 licence하는 결론은 "§3.3의 적용 범위 한정" 방향뿐입니다.')
print( '     "비묶음에서도 동률 유지"는 이 설계로 결론 낼 수 없습니다(5.6% 큰 모델의 라우팅 민감도가 다를 수 있음).')
print( '  3. 측정 범위: untied attention / tied router / tied ln2.')
print( '     [8]의 per-layer independent router 축은 미측정입니다.')
print( '  4. n=3 표본표준편차로 유의성 판정을 쓰지 말고, 효과크기·범위·사전 등록 마진(0.010 nats)만 보고할 것.')
print( '  5. 이 세션의 절대 val loss는 논문 표에 넣지 마세요. 세션 내부 차이(Δ)만 보고합니다.')
print( '  6. 접촉 수는 MoE 모듈 4개 평균이며 pre/post는 상한이 2입니다.')
print( '     루프 모듈만의 값 = (4x − 4)/2, 상한 6. (논문 §3.3의 "2.37 of a possible 6"도 같은 문제입니다.)')
print('=' * 72)

with open(f'{OUT}/s3_untied_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['run', 'seed', 'tie_attn', 'best_val_loss', 'uniq_exp'])
    for s in SEEDS:
        for r in ('moe-loop-st-ut', 'moe-loop-rr-ut'):
            v = best(r, s)
            if v is not None:
                w.writerow([r, s, False, f'{v:.4f}', touches(r, s)])
    for r in ('moe-loop-st', 'moe-loop-rr'):
        v = best(r)
        if v is not None:
            w.writerow([r, 1337, True, f'{v:.4f}', touches(r)])
print(f'\n인계 파일: {OUT}/s3_untied_results.csv')
print('M1 Max로 넘길 것은 이 CSV + model.py diff + 커밋 sha, 그 외에는 없음.')

In [ ]:
# [11] 마무리 — 드라이브 플러시 후 런타임 반납. 반드시 실행.
from google.colab import drive
drive.flush_and_unmount()
print('드라이브 플러시 완료. 런타임을 반납합니다.')
from google.colab import runtime
runtime.unassign()